# S6E9: 6 of 9 Models Add Exactly Zero

You are somewhere around 0.9457. So is everybody else. The top few hundred teams on this
board are packed into a span you could cover with one standard error, and the question in
front of you is which thing to add next: another model family, another block of engineered
features, a bigger stack.

This notebook is about what **not** to add, measured.

I trained nine model families as blend members and built a ten-rung feature-engineering
ladder on top of the best of them. Then I priced every rung and every member against this
board's actual noise floor, which I recompute from `train.csv` in Part 1 rather than quoting.
Four things came out of it.

1. **Six of the nine families contribute exactly zero.** Not "a little". Zero, to six
   decimal places, in a greedy blend cross-fitted outside the fold. Once LightGBM with
   frequency encoding, an XGBoost variant and a plain LightGBM are in the blend, CatBoost,
   HistGradientBoosting, an MLP, a spline GAM, ExtraTrees and a target-encoded XGBoost are
   all assigned zero weight on every fold. Model **diversity did not pay here**.
2. **One encoding change was worth 1.06 standard errors. Everything after it was worth
   0.16.** Ten rungs of feature work, and the first rung is ninety per cent of the ladder.
3. **The rung with the most features is the worst of them.** Adding 78 more engineered
   columns to the 296-column rung, on the same schedule, made the score go *down*.
4. **The plateau is a property of the data, not of my models.** Part 6 is the argument for
   that, and it is the part I care about. This is a synthetic frame: a generator was fitted to
   a small real dataset and sampled to produce 955,236 rows over thirteen columns, and the
   amount of signal in those columns is fixed by the generator, not by me. Part 6 builds a
   staircase out of it. Thirteen cross-fitted lookup tables with no model in them at all get
   most of the way. Trees on the same raw columns add a little. One encoding adds one standard
   error. Everything after that adds nothing a leaderboard could show. Each level of machinery
   buys a small fraction of what the level before it bought, which is what a ceiling looks
   like from underneath, and it is why adding a tenth model family cannot break through.

The fourth point is a claim, so Part 6 tries to falsify it with measurements from
`train.csv` and `test.csv`: a learning curve, an additive-lookup ceiling, an adversarial
validation and a duplicate-row census. If those had come out the other way I would have said
so and dropped the claim. Where they qualify it, I have left the qualification in.

**What this is not.** It is not a model and it produces no submission. Nothing here will move
your public score by itself. It is a measuring instrument, and the thing being measured is
how much headroom is left.

**Credit.** The slow-and-wide LightGBM schedule in rungs 4, 6 and 7 is from a public notebook
by the Kaggle user **najiama**. The eight interaction features and the frequency encoding
underneath the whole ladder are from a public notebook by **evgendvorkin**. The variance
results are Hanley and McNeil (1982) and DeLong, DeLong and Clarke-Pearson (1988).
Competition:
[Playground Series S6E9](https://www.kaggle.com/competitions/playground-series-s6e9).

## The cell you edit

Two knobs matter. `REPLICATION_ROWS` sets the size of the live ladder replication in Part 4:
it is a row subsample, because the full ladder took hours and a kernel has nine. `RESAMPLES`
sets how hard Part 1 works at the noise floor.

Everything else is the competition's own shape and does not need touching.

In [ ]:
CFG = {
    "TARGET": "Will_Buy_EV",
    "ID_COL": "id",
    "POSITIVE": "Yes",

    # ASSUMPTION, not a published figure. I checked the Overview, Evaluation and
    # Data pages on 2026-09-17 and none of them state the public/private split
    # for this competition. 20% is the Playground Series convention. Part 1 shows
    # what the conclusions do across the whole plausible range instead.
    # The row counts below follow from that plus test.csv's own length.
    "PUBLIC_FRACTION": 0.20,

    "SEED": 42,
    "N_SPLITS": 5,          # the fold convention the whole S6E9 library shares
    "RESAMPLES": 30,        # shuffles for the brute-force standard error

    # --- Part 4, the live ladder replication -------------------------------
    # A row subsample. The full ladder ran on all 668,665 rows and took hours;
    # this runs on a slice and reports what the slice actually says, including
    # where it disagrees with the full run.
    "REPLICATION_ROWS": 200000,
    "REPLICATION_FOLDS": 5,
    "REPLICATION_TREES": 700,

    # --- Part 6, the generator evidence ------------------------------------
    "CURVE_FRACTIONS": (0.0625, 0.125, 0.25, 0.5, 1.0),   # exact doublings
    "LOOKUP_BINS": 32,      # quantile bins for the two wide numeric columns
    "LOOKUP_SMOOTH": 50.0,  # pseudo-counts shrinking a cell mean to the prior

    # --- the reference standard error --------------------------------------
    # Measured previously on this competition's own data by DeLong's exact
    # variance and by brute-force resampling, which agreed. Part 1 recomputes
    # it from scratch and compares; this is here so the comparison has a target.
    "SE_REFERENCE": 0.001052,
}

---

## Part 1: The floor everything else is measured against

A leaderboard score is an estimate. It is the AUC of your predictions on one random fifth of
the test rows, and a different fifth would have given a different number. Until you know how
much different, a table of gains is a table of digits.

So this comes first, and every claim in Parts 2 to 6 is quoted as a multiple of it.

I derive it three ways. The first is the closed form everyone reaches for, and on this frame
it is wrong by half.

### The split, from the competition's own files

In [ ]:
import os, gc, json, math, time, warnings, itertools
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
T0 = time.time()

ROOT = os.environ.get("SMOKE_ROOT", "/kaggle/input")
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working")

def find(name):
    for root, _, files in os.walk(ROOT):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

TARGET, ID = CFG["TARGET"], CFG["ID_COL"]
train = pd.read_csv(find("train.csv"))
test  = pd.read_csv(find("test.csv"))

def to_binary(s):
    """The target ships as Yes/No here; other forks have had 0/1 and True/False."""
    if s.dtype.kind in "iufb":
        return s.astype(np.int8).to_numpy()
    v = s.astype(str).str.strip()
    return (v == CFG["POSITIVE"]).astype(np.int8).to_numpy()

SMOKE_ROWS = int(os.environ.get("SMOKE_ROWS", "0"))
if SMOKE_ROWS:
    train = train.sample(SMOKE_ROWS, random_state=CFG["SEED"]).reset_index(drop=True)
    print(f"SMOKE_ROWS={SMOKE_ROWS}: train subsampled. Every number below is a "
          f"small-sample number.\n")

y = to_binary(train[TARGET])
N_TRAIN, N_TEST = len(train), len(test)
PREV = float(y.mean())
N_PUBLIC = round(N_TEST * CFG["PUBLIC_FRACTION"])
N_PRIVATE = N_TEST - N_PUBLIC
FEATS = [c for c in train.columns if c not in (ID, TARGET)]
NUMERIC = [c for c in FEATS if pd.api.types.is_numeric_dtype(train[c])]
CATEG = [c for c in FEATS if c not in NUMERIC]

print("The arithmetic, one line at a time")
print("-" * 64)
print(f"  train rows                            {N_TRAIN:>12,}")
print(f"  test rows                             {N_TEST:>12,}")
print(f"  feature columns                       {len(FEATS):>12}"
      f"   ({len(NUMERIC)} numeric, {len(CATEG)} categorical)")
print(f"  ASSUMED public fraction               {CFG['PUBLIC_FRACTION']:>12.2f}   (not published - see below)")
print(f"  public split rows  = {N_TEST:,} x {CFG['PUBLIC_FRACTION']:.2f}  = {N_PUBLIC:>9,}")
print(f"  private split rows = {N_TEST:,} - {N_PUBLIC:,}     = {N_PRIVATE:>9,}")
print(f"  positive rate in train                {PREV:>12.6f}")
print(f"  implied positives on the public split {round(N_PUBLIC * PREV):>12,}")
print("-" * 64)
print(f"  the private split is {N_PRIVATE / N_PUBLIC:.0f}x the public one, so a standard error")
print(f"  computed on the public split is sqrt({N_PRIVATE / N_PUBLIC:.0f}) = "
      f"{math.sqrt(N_PRIVATE / N_PUBLIC):.1f}x the private one")

Two lines there are inference rather than fact, and both are worth naming before anything is
built on them.

**The 20% is an assumption, and I want to be exact about that.** I went looking for the
published figure on 2026-09-17 and it is not there: the Overview, the Evaluation section and
the Data page say nothing about how test is divided into public and private. Twenty per cent
is the Playground Series convention, not a statement Kaggle makes about this competition. I
had previously described it as published. That was wrong, and this is the correction.

It matters less than you might expect, and the next cell shows why rather than asserting it.
The standard error scales as one over the square root of the split size, so halving the
fraction multiplies it by 1.41 and doubling it divides by the same. Every conclusion in this
notebook is of the form "this gain is smaller than the noise", and a *larger* standard error
only makes those conclusions stronger. The one direction that could hurt is the fraction being
much larger than assumed, so the cell below prices the whole plausible range and you can read
off the answer for whichever value you believe.

**The positive rate comes from train, not from test.** I never see the test labels, so I am
assuming test has roughly train's class balance. For a Playground frame drawn by one
generator in one pass that is a fair assumption, and the next cell shows the standard error
barely moves across any plausible prevalence.

### How much does the unpublished split fraction actually matter?

In [ ]:
# The two headline gains this notebook prices, in raw AUC:
GAIN_RUNG1   = 0.001117   # z0 -> z1, the one encoding change
GAIN_LADDER  = 0.000164   # the whole span of rungs 1..9
GAIN_BLEND   = 0.000103   # nine model families blended vs the best single

print("If the public split is actually ...")
print(f"{'fraction':>9}{'rows':>10}{'SE':>10}{'rung1':>9}{'rungs2-9':>10}{'blend':>8}")
print("-" * 56)
for frac in (0.10, 0.15, 0.20, 0.25, 0.33, 0.50):
    rows = round(N_TEST * frac)
    # SE scales as 1/sqrt(n) from the reference measured at the assumed fraction
    se = CFG["SE_REFERENCE"] * math.sqrt(round(N_TEST * CFG["PUBLIC_FRACTION"]) / rows)
    mark = "  <- assumed" if abs(frac - CFG["PUBLIC_FRACTION"]) < 1e-9 else ""
    print(f"{frac:>9.2f}{rows:>10,}{se:>10.6f}"
          f"{GAIN_RUNG1/se:>8.2f}x{GAIN_LADDER/se:>9.2f}x{GAIN_BLEND/se:>7.2f}x{mark}")
print("-" * 56)
print("Columns 4-6 are each gain in units of that standard error.")
print()
print("Read the table, not my assumption. Across the entire range the whole")
print("feature ladder past rung 1, and the nine-model blend, both stay well")
print("under a quarter of a standard error. Neither becomes visible on a")
print("leaderboard for ANY plausible split size, so nothing in this notebook's")
print("conclusion depends on the 20% being right.")

### Way one: the closed form, and why I stopped trusting it here

In [ ]:
def hanley_se(auc, n_pos, n_neg):
    """Hanley and McNeil (1982): the standard closed-form SE of a single AUC.

    Derived under a binormal / exponential ROC model. q1 and q2 are the two
    conditional variance terms that model implies. It needs only the AUC and the
    two class counts, which is why it is everywhere.
    """
    if n_pos < 1 or n_neg < 1:
        return float("nan")
    q1 = auc / (2 - auc)
    q2 = 2 * auc * auc / (1 + auc)
    var = (auc * (1 - auc)
           + (n_pos - 1) * (q1 - auc * auc)
           + (n_neg - 1) * (q2 - auc * auc)) / (n_pos * n_neg)
    return math.sqrt(max(var, 0.0))

def hanley_at(auc, n, prev=None):
    prev = PREV if prev is None else prev
    n_pos = max(1, round(n * prev))
    return hanley_se(auc, n_pos, n - n_pos)

print(f"{'AUC level':>12} {'SE public':>12} {'SE private':>12}   (Hanley-McNeil)")
for a in (0.9385, 0.9420, 0.9445, 0.9457, 0.9467):
    print(f"{a:>12.4f} {hanley_at(a, N_PUBLIC):>12.6f} {hanley_at(a, N_PRIVATE):>12.6f}")

HANLEY_PUB = hanley_at(0.9457, N_PUBLIC)
print()
print(f"At the level this board is playing at (0.9457), the closed form says the public")
print(f"leaderboard resolves to +/- {HANLEY_PUB:.6f}.")
print()
print("Sensitivity to the prevalence assumption, at AUC 0.9457, public split:")
for p in (0.16, 0.17, PREV, 0.18, 0.19):
    print(f"   prevalence {p:.4f} -> SE {hanley_at(0.9457, N_PUBLIC, p):.6f}")

### An instrument to check the formula against

Hanley-McNeil is a model. It assumes a shape for the ROC curve, and a generated tabular frame
scored by a gradient-booster is under no obligation to have that shape. To check it I need
real predictions against real labels, and the only labelled rows anyone here has are in
`train.csv`.

So the next cell fits a plain LightGBM out of fold: five folds, shuffled, seed 42, which is
the partition the rest of this notebook and most of this board's public notebooks already
share. Thirteen raw columns, nothing engineered, no tuning. It is not trying to be good. It
is trying to be a realistic prediction vector at roughly this board's difficulty, so that the
variance measured on it is the variance you would actually see.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X_PLAIN = train[FEATS].copy()
for c in CATEG:
    X_PLAIN[c] = X_PLAIN[c].astype("category")

FOLDS = np.full(N_TRAIN, -1, dtype=np.int8)
skf = StratifiedKFold(CFG["N_SPLITS"], shuffle=True, random_state=CFG["SEED"])
for k, (_, va) in enumerate(skf.split(X_PLAIN, y)):
    FOLDS[va] = k

PLAIN_PARAMS = dict(objective="binary", n_estimators=600, learning_rate=0.05,
                    num_leaves=63, min_child_samples=40, subsample=0.9,
                    subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                    verbose=-1)

t = time.time()
OOF = np.zeros(N_TRAIN)
for k in range(CFG["N_SPLITS"]):
    tr_i, va_i = np.where(FOLDS != k)[0], np.where(FOLDS == k)[0]
    m = lgb.LGBMClassifier(random_state=CFG["SEED"], n_jobs=-1, **PLAIN_PARAMS)
    m.fit(X_PLAIN.iloc[tr_i], y[tr_i])
    OOF[va_i] = m.predict_proba(X_PLAIN.iloc[va_i])[:, 1]
    del m; gc.collect()
AUC_PLAIN = roc_auc_score(y, OOF)
print(f"plain LightGBM, {len(FEATS)} raw columns, out-of-fold AUC {AUC_PLAIN:.6f}"
      f"   ({time.time() - t:.0f}s)")
print(f"That is the instrument. The top of this board is roughly 0.004 above it, on")
print("features it does not have. It is close enough in difficulty to measure variance on.")

### Way two: DeLong's exact variance

DeLong, DeLong and Clarke-Pearson give the exact non-parametric variance of an AUC. It
assumes nothing about the ROC's shape: it reads the variance off the scores themselves,
through the two sets of placement values. It costs one sort.

In [ ]:
from scipy.stats import rankdata

def delong_var(score, label):
    """Exact non-parametric variance of the AUC (DeLong et al. 1988), via midranks.

    v10[i] is the fraction of negatives that positive i beats; v01[j] the fraction
    of positives that negative j loses to. Var = var(v10)/m + var(v01)/n.
    """
    x, z = score[label == 1], score[label == 0]
    m, n = len(x), len(z)
    r_all = rankdata(np.concatenate([x, z]))
    v10 = (r_all[:m] - rankdata(x)) / n
    v01 = 1.0 - (r_all[m:] - rankdata(z)) / m
    return float(v10.mean()), float(v10.var(ddof=1) / m + v01.var(ddof=1) / n)

auc_full, var_full = delong_var(OOF, y)
DELONG_PUB = math.sqrt(var_full * N_TRAIN / N_PUBLIC)

rng = np.random.default_rng(CFG["SEED"])
direct = []
if N_TRAIN >= N_PUBLIC:
    for _ in range(5):
        idx = rng.choice(N_TRAIN, N_PUBLIC, replace=False)
        direct.append(math.sqrt(delong_var(OOF[idx], y[idx])[1]))

print(f"DeLong on all {N_TRAIN:,} out-of-fold rows: AUC {auc_full:.6f}, "
      f"SE {math.sqrt(var_full):.6f}")
print(f"  scaled to a {N_PUBLIC:,}-row split (SE goes as 1/sqrt(n)): {DELONG_PUB:.6f}")
if direct:
    print(f"  computed directly on five random {N_PUBLIC:,}-row samples: "
          + ", ".join(f"{s:.6f}" for s in direct))
    print(f"  mean of those five: {np.mean(direct):.6f}   (the scaling holds)")
else:
    print(f"  the direct check needs {N_PUBLIC:,} rows to draw from and this run has "
          f"{N_TRAIN:,}, so it is skipped. On the full data it runs.")
print()
print(f"Hanley says {hanley_at(auc_full, N_PUBLIC):.6f} at this same AUC. "
      f"DeLong says {DELONG_PUB:.6f}.")
print(f"Ratio: Hanley is {hanley_at(auc_full, N_PUBLIC) / DELONG_PUB:.2f}x larger.")

### Way three: stop deriving and resample

Two formulas disagreeing is not a result. The arbiter is brute force: cut the out-of-fold
rows into chunks the size of the public split, score each chunk, and look at the spread.

Each shuffle yields `floor(N_train / n_public)` **disjoint** chunks, so the chunks inside one
shuffle are genuinely independent of each other. I pool the within-shuffle variances across
shuffles rather than taking one variance over all of them, which would count the overlap
between shuffles as new information.

One correction is easy to forget. Drawing `n_public` rows without replacement from a fixed
pool gives slightly *less* spread than drawing from an unlimited population. The
finite-population factor is `sqrt(1 - n_public/N_train)`; dividing it out is worth a few per
cent and I would rather show it than quietly absorb it.

In [ ]:
def resample_aucs(vec, n_split, shuffles, seed):
    """AUC of `vec` on each of shuffles x floor(N/n_split) disjoint chunks."""
    chunks = N_TRAIN // n_split
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(shuffles):
        perm = rng.permutation(N_TRAIN)
        for i in range(chunks):
            idx = perm[i * n_split:(i + 1) * n_split]
            out.append(roc_auc_score(y[idx], vec[idx]))
    return np.asarray(out), chunks

t = time.time()
CH, N_CHUNKS = resample_aucs(OOF, N_PUBLIC, CFG["RESAMPLES"], CFG["SEED"])
FPC = math.sqrt(1 - N_PUBLIC / N_TRAIN)
n_est = CFG["RESAMPLES"] * N_CHUNKS
within = [np.var(CH[s * N_CHUNKS:(s + 1) * N_CHUNKS], ddof=1)
          for s in range(CFG["RESAMPLES"])]
EMPIRICAL_PUB = math.sqrt(float(np.mean(within))) / FPC

print(f"{CFG['RESAMPLES']} shuffles x {N_CHUNKS} disjoint chunks of {N_PUBLIC:,} rows "
      f"= {n_est} scored splits   ({time.time() - t:.0f}s)")
print(f"  chunk AUCs: mean {CH.mean():.6f}, min {CH.min():.6f}, max {CH.max():.6f}  "
      f"(spread {CH.max() - CH.min():.6f})")
print(f"  pooled within-shuffle SD           {math.sqrt(float(np.mean(within))):.6f}")
print(f"  finite-population factor           sqrt(1 - {N_PUBLIC:,}/{N_TRAIN:,}) = {FPC:.4f}")
print(f"  corrected empirical SE             {EMPIRICAL_PUB:.6f}")
print()
print(f"{'method':<34}{'public SE':>12}{'vs DeLong':>12}")
print("-" * 58)
for lbl, v in (("Hanley-McNeil closed form", hanley_at(auc_full, N_PUBLIC)),
               ("DeLong exact variance", DELONG_PUB),
               (f"resampling, {n_est} splits", EMPIRICAL_PUB)):
    print(f"{lbl:<34}{v:>12.6f}{v / DELONG_PUB:>11.2f}x")

DeLong and brute force agree closely. Hanley is half again too big, and it has a cause: its
`q1` and `q2` terms come from an exponential ROC model, which is a poor description of a
heavily-tied gradient-boosting score distribution on a generated tabular frame. On this board
it is a conservative bound rather than an estimate.

The small residual gap between DeLong and the resampling has a cause too. DeLong conditions
on the number of positives; my resampling lets the class balance wobble from chunk to chunk
the way a real random split does. That extra binomial wobble is real and is part of what the
private split will do to you, so I take the resampled number as the honest one.

The next cell also holds this run's floor up against the reference figure of 0.001052 that
the tables in Parts 2 and 3 are quoted against. I keep the reference so those tables stay
comparable with the run they came from, and it is worth noticing which way that cuts: the
resampled floor measured here is the *larger* of the two, and dividing a gain by a larger
floor makes the gain look smaller. Nothing in this notebook depends on my having picked the
flattering number.

In [ ]:
SIGMA_PUB = EMPIRICAL_PUB
SIGMA_PRV = SIGMA_PUB * math.sqrt(N_PUBLIC / N_PRIVATE)
SE = CFG["SE_REFERENCE"]

print(f"  public-split SE    {SIGMA_PUB:.6f}   (on {N_PUBLIC:,} rows, measured here)")
print(f"  private-split SE   {SIGMA_PRV:.6f}   (on {N_PRIVATE:,} rows)")
print(f"  reference SE       {SE:.6f}   (earlier run, same two methods)")
print(f"  this run vs reference: {SIGMA_PUB / SE:.3f}x")
print()
print("Every gain from here on is quoted as a multiple of the reference public SE, so that")
print("the tables stay comparable with the earlier work they came from. Substitute this")
print(f"run's own {SIGMA_PUB:.6f} if you prefer; it changes the digits, not one conclusion.")

def in_se(delta, sigma=None):
    return delta / (SE if sigma is None else sigma)

assert 0.0006 < SIGMA_PUB < 0.0018, SIGMA_PUB

---

## Part 2: Nine model families, and what each one was worth

This is the first of two tables the notebook cannot recompute. Nine families, each trained
five-fold out of fold on the same seed-42 partition Part 1 just built, then blended by a
greedy weight search cross-fitted outside the fold so the weights never see the fold they are
scored on.

**Source: a completed full run on all 668,665 rows.** Not reproduced in this kernel; the
nine members took hours. The recipe for each is in the table so you can check any of them.

In [ ]:
# -------------------------------------------------------------------- FULL RUN
# Out-of-fold AUC of each blend member. 5-fold StratifiedKFold(shuffle=True,
# random_state=42) over train.csv in original file order, target 'Yes' -> 1.
# These nine numbers come from a completed full run, NOT from this kernel.
MEMBERS = [
    ("a", "LightGBM, raw + quantisation digits",              0.943742),
    ("b", "XGBoost + target encoding",                        0.941447),
    ("c", "CatBoost",                                         0.941177),
    ("d", "HistGradientBoosting",                             0.941612),
    ("e", "MLP",                                              0.938387),
    ("f", "Spline GAM",                                       0.939176),
    ("g", "ExtraTrees",                                       0.938105),
    ("h", "LightGBM + frequency encoding",                    0.944507),
    ("i", "XGBoost, evgendvorkin-style ladder",               0.944407),
]
# Marginal contribution: how much the nested greedy blend loses if that member
# is removed from the pool. Same full run.
MARGINAL = {"a": 0.000014, "b": 0.0, "c": 0.0, "d": 0.0, "e": 0.0,
            "f": 0.0, "g": 0.0, "h": 0.000140, "i": 0.000102}
BLEND_AUC        = 0.944609
BEST_SINGLE_AUC  = 0.944507
BLEND_MEAN_DELTA = 0.000103
BLEND_DELTA_SD   = 0.00005
BLEND_T          = 4.57

print("FULL RUN, 668,665 rows, 5 folds, seed 42. Not recomputed in this kernel.")
print()
print(f"{'':>3}{'family':<40}{'OOF AUC':>11}{'vs best':>11}{'marginal':>11}{'in SE':>9}")
print("-" * 85)
for tag, what, auc in sorted(MEMBERS, key=lambda r: -r[2]):
    mc = MARGINAL[tag]
    print(f"{tag:>3}  {what:<39}{auc:>10.6f}{auc - BEST_SINGLE_AUC:>11.6f}"
          f"{mc:>11.6f}{in_se(mc):>9.3f}")
print("-" * 85)
n_zero = sum(1 for v in MARGINAL.values() if v == 0.0)
print(f"{'':>3}{'members with zero weight on EVERY fold':<40}{n_zero:>10} of {len(MEMBERS)}")
print()
print(f"nested greedy blend            {BLEND_AUC:.6f}")
print(f"best single member (h)         {BEST_SINGLE_AUC:.6f}")
print(f"blend gain                     {BLEND_MEAN_DELTA:+.6f}  "
      f"= {in_se(BLEND_MEAN_DELTA):.3f} SE")
print(f"  per-fold sd {BLEND_DELTA_SD:.6f}, t-like {BLEND_T:.2f}, all five folds positive")

Read the marginal column, because that is the surprise.

**Six of the nine families are worth exactly zero.** b, c, d, e, f and g take zero weight on
every fold of the nested search. Not a small weight. Zero. The blend is a, h and i, and a
itself is worth 0.000014, which is about a seventy-fifth of a standard error.

This is not what the diversity literature promises and it is not what I expected when I built
nine families on purpose. A CatBoost, a HistGB, a neural net, a spline GAM and an ExtraTrees
are about as different in inductive bias as you can get with off-the-shelf parts, and they
still had nothing to add once h and i were present. Their **errors are not independent
enough to matter**. They are all approximating the same function and missing it in the same
places.

And the blend itself, the whole nine-member apparatus, is worth `+0.000103`. It is real:
positive on all five folds, t-like 4.57, so I am not claiming it is noise in the statistical
sense. But it is one tenth of one standard error. On a leaderboard you would never see it.

Part 4 runs the same greedy search live, inside this kernel, over six members on a
subsample, so you can watch the mechanism rather than take the table's word for it.

That distinction matters and I want to be exact about it. A gain can be **statistically
real** and **practically invisible** at the same time. Cross-validation with five paired
folds can resolve a difference far below what a single 57,000-row public split can resolve.
Everything in this notebook that I call invisible is invisible *on a leaderboard*, which is
the only place it gets paid.

---

## Part 3: Ten rungs of feature engineering

Second table from the full run. Every rung is built on member h's 154 columns (7 numeric
plus 6 categorical plus 56 quantisation digits plus 8 interactions, each of those 77 with a
frequency-encoded twin) and adds exactly one thing, so a feature effect is never confounded
with a schedule effect.

**Source: a completed full run on all 668,665 rows**, same folds as Part 2. Part 4 replicates
the two ends of this ladder live at reduced scale.

In [ ]:
# -------------------------------------------------------------------- FULL RUN
# Ten rungs, each on the same 5-fold seed-42 partition. rung 0 is member h
# exactly and doubles as a reproduction check. FULL RUN numbers; not recomputed.
LADDER = [
    (0, 154, 0.944507, "member h exactly, reproduction check"),
    (1, 292, 0.945624, "+ dual sklearn TargetEncoder (smooth auto and 10.0, cv=5) over "
                       "numerics, categoricals and digits as string categories"),
    (2, 305, 0.945578, "+ target means per value from the real 10k "
                       "EV_Adoption_and_Range_Anxiety dataset"),
    (3, 309, 0.945602, "+ four hand flags: income==30000, income>=170537, "
                       "38000<=income<=42000, env_concern==1"),
    (4, 309, 0.945741, "+ najiama's slow-and-wide schedule, scaled to lr 0.015 / 20k trees "
                       "/ max_bin 511 / patience 400"),
    (5, 296, 0.945612, "best subset: h + dual target encoding + the four flags, WITHOUT the "
                       "origin-dataset means rung 2 measured as a loss"),
    (6, 296, 0.945731, "best subset + that schedule, scaled"),
    (7, 296, 0.945724, "best subset + najiama's literal schedule "
                       "(lr 0.005, 40k trees, max_bin 1024, patience 700)"),
    (8, 374, 0.945577, "best subset + PAIR target encoding over all 78 raw-feature pairs"),
    (9, 351, 0.945662, "best subset + PAIR target encoding over supported pairs only "
                       "(both operands <= 50 distinct)"),
]
BASE_AUC = LADDER[0][2]
RUNG = {z: auc for z, _, auc, _ in LADDER}

print("FULL RUN, 668,665 rows, 5 folds, seed 42. Not recomputed in this kernel.")
print(f"Every gain quoted against the reference public SE = {SE:.6f}.")
print()
print(f"{'rung':>5}{'feats':>7}{'OOF AUC':>11}{'vs rung 0':>11}{'in SE':>8}"
      f"{'vs prev':>10}  what it adds")
print("-" * 118)
prev = None
for z, nf, auc, what in LADDER:
    step = "" if prev is None else f"{auc - prev:>+10.6f}"
    print(f"{'z' + str(z):>5}{nf:>7}{auc:>11.6f}{auc - BASE_AUC:>+11.6f}"
          f"{in_se(auc - BASE_AUC):>8.3f}{step:>10}  {what[:58]}")
    prev = auc
print("-" * 118)

POST = [auc for z, _, auc, _ in LADDER if z >= 1]
SPAN_POST = max(POST) - min(POST)
FIRST_RUNG = RUNG[1] - RUNG[0]
BEST_Z = max(LADDER, key=lambda r: r[2])
WIDEST = max(LADDER, key=lambda r: r[1])

print(f"  rung 0 -> rung 1, one change          {FIRST_RUNG:+.6f}  "
      f"= {in_se(FIRST_RUNG):.3f} SE")
print(f"  rungs 1..9, best minus worst          {SPAN_POST:+.6f}  "
      f"= {in_se(SPAN_POST):.3f} SE")
print(f"  best rung is z{BEST_Z[0]} at {BEST_Z[2]:.6f} with {BEST_Z[1]} features")
print(f"  widest rung is z{WIDEST[0]} with {WIDEST[1]} features at {WIDEST[2]:.6f}, "
      f"which is rank {sorted(POST, reverse=True).index(WIDEST[2]) + 1} of 9 among rungs 1-9")

### The same table as a picture

In [ ]:
import matplotlib.pyplot as plt

zz = [z for z, _, _, _ in LADDER]
aa = [auc for _, _, auc, _ in LADDER]
nn = [nf for _, nf, _, _ in LADDER]

fig, ax = plt.subplots(figsize=(10.5, 5.4))
ax.axhspan(RUNG[1] - SE, RUNG[1] + SE, color="#c8dcf0", alpha=0.55, zorder=0,
           label=f"+/- 1 public SE around rung 1  ({SE:.6f})")
ax.axhline(RUNG[1], color="#4a7fb5", lw=1.0, ls="--", zorder=1)
ax.plot(zz, aa, "-o", color="#2f4858", lw=1.6, ms=6, zorder=3)
ax.plot([0], [RUNG[0]], "o", color="#c1442e", ms=11, zorder=4,
        label="rung 0 (member h), the only point outside the band")
for z, nf, auc, _ in LADDER:
    ax.annotate(f"{nf}", (z, auc), textcoords="offset points", xytext=(0, -15),
                ha="center", fontsize=8, color="#666")
ax.set_xticks(zz)
ax.set_xlabel("rung  (number under each point = feature count)")
ax.set_ylabel("out-of-fold AUC")
ax.set_title("Ten rungs of feature engineering against this board's noise floor\n"
             "FULL RUN, 668,665 rows, 5 folds, seed 42", fontsize=11)
ax.legend(loc="lower right", fontsize=9, framealpha=0.95)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

inside = [z for z, _, auc, _ in LADDER if abs(auc - RUNG[1]) <= SE]
outside = [z for z, _, auc, _ in LADDER if abs(auc - RUNG[1]) > SE]
print(f"rungs inside the +/- 1 SE band around rung 1: {inside}")
print(f"rungs outside it:                            {outside}")
print()
print(f"rung 0 sits {abs(RUNG[0] - RUNG[1]) / SE:.2f} SE below rung 1 - outside the band, "
      f"but only just.")
print("Everything from rung 2 to rung 9 is inside it. Eight rungs of work, one band.")

The band is one public standard error either side of rung 1. Rung 0 falls below it; every
rung from 2 to 9 falls inside it.

Three of those rungs are worth pointing at individually.

**Rung 2 went backwards.** Adding target means from the real 10,000-row
`EV_Adoption_and_Range_Anxiety` dataset that this competition's generator was fitted on *lost*
score. Rungs 5 to 9 drop that ingredient for exactly that reason. The original data is
supposed to be the closest thing to ground truth available, and feeding it in as per-value
target means made things worse.

**Rung 8 has the most features and the worst score of the nine post-base rungs.** It
target-encodes all 78 pairs of raw features, taking the frame from 296 columns to 374. Rung 9
keeps only the pairs whose operands are both narrow, a support condition rather than a score-based
selection, so it cannot leak, and recovers most of the loss without getting back above the
best rung. Pairing two wide columns produces more keys than the frame has rows, so the inner
cross-fit shrinks nearly all of them to the prior and the model is left scanning noise.

**The schedule rungs are the only ones after rung 1 that reliably help.** najiama's
slow-and-wide LightGBM schedule shows up in rungs 4, 6 and 7 and is worth roughly an eighth of
a standard error each time. Rung 7 runs his literal parameters, learning rate 0.005, 40,000
trees and `max_bin` 1024, and lands a whisker *below* the scaled version in rung 6. Several
times the wall time for a difference of seven parts in a million.

---

## Part 4: The same ladder, live, at a size a kernel can afford

A table I cannot recompute is a table you have to take on trust. So here are the ladder's two
ends rebuilt from scratch inside this notebook, on a row subsample, with the schedule held
fixed so only the features move:

- **R0**: member h's 154 columns: raw numerics, categoricals, 56 quantisation digits, the 8
  interactions, and a frequency-encoded twin of all 77. This is rung 0.
- **R1**: R0 plus the dual target encoding. This is rung 1, the one change that paid.
- **R2**: R1 plus the four flags plus pair target encoding over the supported pairs. This is
  rung 9, the "throw more features at it" end.

Three differences from the full run, all of which make this the *harsher* test and none of
which I am hiding:

1. It runs on a subsample, so its own standard error is larger and its deltas are noisier.
2. It uses a fixed tree count with **no early stopping on the validation fold**. The full-run
   ladder inherited early stopping on the fold it is scored on, which makes every full-run
   number mildly optimistic in the same direction. This replication has no such advantage.
3. Frequency encoding is computed once over all of `train.csv` and `test.csv` rather than per
   fold. It reads no labels, so this is not leakage; it is the transductive step the recipe
   has always had, done once instead of five times.

If the shape does not reproduce, that is a finding and it goes in the write-up.

In [ ]:
from sklearn.preprocessing import TargetEncoder

DIGIT_POSITIONS = tuple(range(-4, 4))          # 10^-4 .. 10^3, member a's range
# The 8 interactions are evgendvorkin's, chosen by hand on this board.
INTERACTIONS = (
    ("Env_x_Subsidy",        "mul",  "Environmental_Concern_Level", "Subsidy_Available"),
    ("Env_x_Income",         "mul",  "Environmental_Concern_Level", "Annual_Income_USD"),
    ("Subsidy_x_Income",     "mul",  "Subsidy_Available",           "Annual_Income_USD"),
    ("Env_x_Commute",        "mul",  "Environmental_Concern_Level", "Daily_Commute_km"),
    ("HomeCharge_x_Subsidy", "mul",  "Home_Charging_Possible",      "Subsidy_Available"),
    ("RangeAnx_x_Subsidy",   "mul",  "Range_Anxiety_Level",         "Subsidy_Available"),
    ("Income_per_Env",       "div1", "Annual_Income_USD",           "Environmental_Concern_Level"),
    ("Commute_per_Env",      "div1", "Daily_Commute_km",            "Environmental_Concern_Level"),
)

def digit_frame(df):
    """digit_k(x) = floor(x / 10**k) mod 10, int8. Fit-free, so it cannot leak."""
    out = {}
    for c in NUMERIC:
        x = df[c].to_numpy(np.float64)
        for k in DIGIT_POSITIONS:
            out[f"{c}__d{k}"] = np.mod(np.floor(x / (10.0 ** k)), 10.0).astype(np.int8)
    return pd.DataFrame(out, index=df.index)

def coded(df, vocab):
    """Numerics as they are, categoricals as integer codes from a fixed vocabulary.

    float32 rather than float64: LightGBM bins to 255 buckets anyway, so the extra
    mantissa buys nothing and the frame is half the size.
    """
    out = {c: df[c].to_numpy(np.float32) for c in NUMERIC}
    for c in CATEG:
        out[c] = pd.Categorical(df[c].astype(str),
                                categories=vocab[c]).codes.astype(np.float32)
    return pd.DataFrame(out, index=df.index)

def interaction_frame(o):
    out = {}
    for name, op, left, right in INTERACTIONS:
        a = o[left].to_numpy(np.float64)
        b = o[right].to_numpy(np.float64)
        out[name] = a * b if op == "mul" else a / (b + 1.0)
    return pd.DataFrame(out, index=o.index)

def build_77(df, vocab):
    """The 77 pre-frequency columns: 7 numeric + 6 categorical + 56 digit + 8 interaction."""
    o = coded(df, vocab)
    return pd.concat([o, digit_frame(df), interaction_frame(o)], axis=1)

t = time.time()
VOCAB = {c: pd.Index(pd.unique(pd.concat([train[c], test[c]]).astype(str))) for c in CATEG}
B_TR = build_77(train, VOCAB)
B_TE = build_77(test, VOCAB)
print(f"77-column base frame built: train {B_TR.shape}, test {B_TE.shape}  "
      f"({time.time() - t:.0f}s)")

t = time.time()
freq_tr = {}
for c in B_TR.columns:
    v = np.concatenate([B_TR[c].to_numpy(), B_TE[c].to_numpy()])
    codes, uniq = pd.factorize(v, use_na_sentinel=False)
    counts = np.bincount(codes, minlength=len(uniq)).astype(np.float32) / len(codes)
    freq_tr[f"{c}_freq"] = counts[codes[:len(B_TR)]]
B_TR = pd.concat([B_TR, pd.DataFrame(freq_tr, index=B_TR.index)], axis=1)
del B_TE, freq_tr; gc.collect()
print(f"frequency twins added over concat(train, test) = {N_TRAIN + N_TEST:,} rows: "
      f"{B_TR.shape[1]} columns  ({time.time() - t:.0f}s)")
assert B_TR.shape[1] == 154, B_TR.shape
print("154 columns, matching member h's feature count exactly.")

In [ ]:
def te_source(df):
    """What the dual TargetEncoder sees: the 13 raw columns and their 56 digits,
    each treated as a bare category. Not the interactions, not the frequency
    columns (that would be encoding an encoding).

    The full run passed these as strings, following najiama. I pass integer codes
    instead. sklearn's TargetEncoder treats any input column as categorical, so
    one code per distinct value is the same partition as one string per distinct
    value, and it is several times faster on 69 columns of 250,000 rows.
    """
    src = pd.concat([df[FEATS], digit_frame(df)], axis=1)
    out = {f"{c}_cat": pd.factorize(src[c], use_na_sentinel=False)[0].astype(np.int32)
           for c in src.columns}
    return pd.DataFrame(out, index=df.index)

NARROW = [c for c in FEATS if train[c].nunique() <= 50]
PAIRS = list(itertools.combinations(NARROW, 2))
print(f"narrow columns (<= 50 distinct): {len(NARROW)} of {len(FEATS)} -> {len(PAIRS)} pairs")
print(f"wide columns excluded: {[c for c in FEATS if c not in NARROW]}")

R_ROWS = min(CFG["REPLICATION_ROWS"], N_TRAIN)
rng = np.random.default_rng(CFG["SEED"])
SUB = np.sort(rng.choice(N_TRAIN, R_ROWS, replace=False)) if R_ROWS < N_TRAIN \
      else np.arange(N_TRAIN)
y_r = y[SUB]
raw_r = train.iloc[SUB].reset_index(drop=True)
base_r = B_TR.iloc[SUB].reset_index(drop=True)
TE_SRC = te_source(raw_r)
del B_TR; gc.collect()          # only the subsample is needed from here on

RF = np.full(R_ROWS, -1, dtype=np.int8)
for k, (_, va) in enumerate(StratifiedKFold(
        CFG["REPLICATION_FOLDS"], shuffle=True, random_state=CFG["SEED"]).split(base_r, y_r)):
    RF[va] = k

R_PARAMS = dict(objective="binary", n_estimators=CFG["REPLICATION_TREES"],
                learning_rate=0.05, num_leaves=63, min_child_samples=40,
                subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                reg_lambda=1.0, max_bin=255, verbose=-1)

print(f"\nreplication: {R_ROWS:,} rows, {CFG['REPLICATION_FOLDS']} folds, "
      f"{CFG['REPLICATION_TREES']} trees, no early stopping")

In [ ]:
def pair_codes(df, vocab):
    return {c: df[c].map(vocab[c]).fillna(-1).astype(np.int64).to_numpy() for c in NARROW}

def build_arm(arm, tr_i, va_i):
    """R0 = 154 columns; R1 = + dual target encoding; R2 = + flags + pair TE."""
    tr, va = base_r.iloc[tr_i], base_r.iloc[va_i]
    if arm == "R0":
        return tr, va
    add_tr, add_va = [], []
    s_tr, s_va = TE_SRC.iloc[tr_i], TE_SRC.iloc[va_i]
    for tag, smooth in (("auto", "auto"), ("s10", 10.0)):
        enc = TargetEncoder(target_type="binary", smooth=smooth, cv=5,
                            shuffle=True, random_state=CFG["SEED"])
        names = [f"{c}_TE_{tag}" for c in s_tr.columns]
        add_tr.append(pd.DataFrame(enc.fit_transform(s_tr, y_r[tr_i]).astype(np.float32),
                                   columns=names, index=tr.index))
        add_va.append(pd.DataFrame(enc.transform(s_va).astype(np.float32),
                                   columns=names, index=va.index))
        del enc; gc.collect()
    if arm == "R2":
        for bucket, idx, frame in ((add_tr, tr_i, tr), (add_va, va_i, va)):
            inc = raw_r["Annual_Income_USD"].to_numpy()[idx]
            env = raw_r["Environmental_Concern_Level"].to_numpy()[idx]
            bucket.append(pd.DataFrame({
                "is_30k_spike":         (inc == 30000.0).astype(np.int8),
                "is_millionaire_cliff": (inc >= 170537.0).astype(np.int8),
                "is_dead_zone":         ((inc >= 38000.0) & (inc <= 42000.0)).astype(np.int8),
                "is_env_hater":         (env == 1).astype(np.int8),
            }, index=frame.index))
        vocab = {c: {v: i for i, v in enumerate(pd.unique(raw_r[c].iloc[tr_i]))}
                 for c in NARROW}
        cd_tr, cd_va = pair_codes(raw_r.iloc[tr_i], vocab), pair_codes(raw_r.iloc[va_i], vocab)
        names = [f"{a}__{b}_pTE" for a, b in PAIRS]
        def src(cd, idx):
            return pd.DataFrame({f"{a}__{b}": (cd[a] + 1) * (len(vocab[b]) + 1) + (cd[b] + 1)
                                 for a, b in PAIRS}, index=idx)
        p_tr, p_va = src(cd_tr, tr.index), src(cd_va, va.index)
        enc = TargetEncoder(target_type="binary", smooth=10.0, cv=5,
                            random_state=CFG["SEED"])
        add_tr.append(pd.DataFrame(enc.fit_transform(p_tr, y_r[tr_i]).astype(np.float32),
                                   columns=names, index=tr.index))
        add_va.append(pd.DataFrame(enc.transform(p_va).astype(np.float32),
                                   columns=names, index=va.index))
        del p_tr, p_va, enc, cd_tr, cd_va; gc.collect()
    return pd.concat([tr] + add_tr, axis=1), pd.concat([va] + add_va, axis=1)

REP = {}
for arm in ("R0", "R1", "R2"):
    t = time.time()
    oof = np.zeros(R_ROWS)
    n_feat = None
    for k in range(CFG["REPLICATION_FOLDS"]):
        tr_i, va_i = np.where(RF != k)[0], np.where(RF == k)[0]
        tb, vb = build_arm(arm, tr_i, va_i)
        n_feat = tb.shape[1]
        m = lgb.LGBMClassifier(random_state=CFG["SEED"], n_jobs=-1, **R_PARAMS)
        m.fit(tb, y_r[tr_i])
        oof[va_i] = m.predict_proba(vb)[:, 1]
        del tb, vb, m; gc.collect()
    REP[arm] = dict(auc=roc_auc_score(y_r, oof), n=n_feat,
                    secs=time.time() - t, oof=oof)
    print(f"  {arm}: {n_feat:>4} features   OOF AUC {REP[arm]['auc']:.6f}   "
          f"({REP[arm]['secs']:.0f}s)")

In [ ]:
# The replication's own standard error, on its own row count. Smaller sample,
# larger SE - quoting the full-run SE here would flatter the result.
_, rep_var = delong_var(REP["R0"]["oof"], y_r)
SE_REP = math.sqrt(rep_var * R_ROWS / N_PUBLIC)

del base_r, TE_SRC; gc.collect()

print(f"replication standard error on a {N_PUBLIC:,}-row split, DeLong on "
      f"{R_ROWS:,} rows: {SE_REP:.6f}")
print(f"  (Part 1 measured {SIGMA_PUB:.6f} on all {N_TRAIN:,} rows)")
print()
print(f"{'step':<16}{'feats':>7}{'live delta':>13}{'in live SE':>12}"
      f"{'full-run delta':>16}{'in ref SE':>11}")
print("-" * 76)
COMPARE = [("R0 -> R1", "R0", "R1", RUNG[1] - RUNG[0], REP["R1"]["n"]),
           ("R1 -> R2", "R1", "R2", RUNG[9] - RUNG[1], REP["R2"]["n"])]
AGREE = {}
for label, a, b, full_delta, nf in COMPARE:
    live = REP[b]["auc"] - REP[a]["auc"]
    AGREE[label] = (live, full_delta)
    print(f"{label:<16}{nf:>7}{live:>+13.6f}{live / SE_REP:>12.3f}"
          f"{full_delta:>+16.6f}{in_se(full_delta):>11.3f}")
print("-" * 76)
print(f"{'R0 -> R2':<16}{REP['R2']['n']:>7}"
      f"{REP['R2']['auc'] - REP['R0']['auc']:>+13.6f}"
      f"{(REP['R2']['auc'] - REP['R0']['auc']) / SE_REP:>12.3f}"
      f"{RUNG[9] - RUNG[0]:>+16.6f}{in_se(RUNG[9] - RUNG[0]):>11.3f}")
print()
for label, (live, full_delta) in AGREE.items():
    sign = "SAME sign" if (live > 0) == (full_delta > 0) else "OPPOSITE sign"
    ratio = f"{abs(live / full_delta):.2f}x the full-run size" if full_delta else "n/a"
    print(f"  {label}: live {live:+.6f} vs full run {full_delta:+.6f} - {sign}, "
          f"live is {ratio}")

### What the replication said

Read the two columns against each other rather than the absolute AUCs, which are lower here
simply because the model saw a fraction of the rows and stopped at a fixed tree count.

The claim being tested is a shape, not a number: **one encoding is worth roughly a standard
error, and piling features on top of it is worth a fraction of one.** The comparison printed
above is the evidence for or against that shape, at a size you can rerun in a fork.

On the run shipped with this notebook, the shape held and the second half held harder than I
expected. The encoding step came back positive and dominant, at a little over half its
full-run size. Target encoding is the one ingredient here that should shrink on a smaller
frame, because every category has fewer rows to estimate its mean from, so a smaller gain at
a third of the data is what the mechanism predicts.

The pile-on step came back **negative**, where the full run had it at a barely positive
+0.000038. Both of those numbers live inside the 0.16-SE band that rungs 1 to 9 share, so
they are two draws from the same puddle rather than a contradiction. I am not going to dress
that up as a confirmation. It is the sign flipping on a quantity too small to have a stable
sign, which is the notebook's own thesis arriving as an inconvenience rather than as a
result.

### And the blend, live, with six members instead of nine

Part 2's headline is the one thing a reader cannot check from a table, so here is the same
greedy nested search run inside this kernel over six members: the three ladder arms above,
plus three genuinely different families fitted on the plain thirteen columns. Same subsample,
same folds.

The mechanism is what matters. For each fold the weights are chosen greedily on the *other*
four folds and then scored on the held-out one, so no member is ever weighted using the rows
it is judged on. A member that is never chosen has a marginal contribution of exactly zero,
which is where the full run's six zeroes came from.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from collections import Counter

X_R = X_PLAIN.iloc[SUB].reset_index(drop=True)
X_R_NUM = pd.get_dummies(X_R, columns=CATEG, dtype=np.float32)
print(f"plain frame for the extra families: {X_R_NUM.shape}")

def oof_of(make, frame):
    out = np.zeros(R_ROWS)
    for k in range(CFG["REPLICATION_FOLDS"]):
        tr_i, va_i = np.where(RF != k)[0], np.where(RF == k)[0]
        m = make()
        m.fit(frame.iloc[tr_i], y_r[tr_i])
        out[va_i] = m.predict_proba(frame.iloc[va_i])[:, 1]
        del m; gc.collect()
    return out

EXTRA = {}
for name, make in (
    ("extratrees", lambda: ExtraTreesClassifier(n_estimators=200, min_samples_leaf=50,
                                                n_jobs=4, random_state=CFG["SEED"])),
    ("histgb",     lambda: HistGradientBoostingClassifier(
                       max_iter=400, learning_rate=0.06, max_leaf_nodes=63,
                       min_samples_leaf=40, random_state=CFG["SEED"])),
    ("logistic",   lambda: make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=1000, C=1.0))),
):
    t = time.time()
    EXTRA[name] = oof_of(make, X_R_NUM)
    print(f"  {name:<12} OOF AUC {roc_auc_score(y_r, EXTRA[name]):.6f}   "
          f"({time.time() - t:.0f}s)")

POOL = {"R0_154feats": REP["R0"]["oof"], "R1_292feats": REP["R1"]["oof"],
        "R2_351feats": REP["R2"]["oof"], **EXTRA}
RANKED = {k: rankdata(v) / len(v) for k, v in POOL.items()}

In [ ]:
GREEDY_ROUNDS = 15

def greedy(members, idx):
    """Forward greedy averaging with replacement, fitted on `idx` rows only."""
    cur = np.zeros(len(idx))
    picks = []
    for _ in range(GREEDY_ROUNDS):
        best_name, best_auc = None, -np.inf
        for nm in members:
            cand = (cur * len(picks) + RANKED[nm][idx]) / (len(picks) + 1)
            a = roc_auc_score(y_r[idx], cand)
            if a > best_auc:
                best_auc, best_name = a, nm
        picks.append(best_name)
        cur = (cur * (len(picks) - 1) + RANKED[best_name][idx]) / len(picks)
    c = Counter(picks)
    return {nm: c[nm] / GREEDY_ROUNDS for nm in members if c[nm]}

def nested_blend(members):
    """Weights fitted on the other folds, scored on the held-out fold."""
    blended, single, per_fold = np.zeros(R_ROWS), np.zeros(R_ROWS), []
    for k in range(CFG["REPLICATION_FOLDS"]):
        inner, outer = np.where(RF != k)[0], np.where(RF == k)[0]
        w = greedy(members, inner)
        blended[outer] = sum(wt * RANKED[nm][outer] for nm, wt in w.items())
        best_nm = max(members, key=lambda nm: roc_auc_score(y_r[inner], RANKED[nm][inner]))
        single[outer] = RANKED[best_nm][outer]
        per_fold.append((k, w, best_nm))
    return roc_auc_score(y_r, blended), roc_auc_score(y_r, single), per_fold

t = time.time()
members = list(POOL)
B_AUC, S_AUC, PER_FOLD = nested_blend(members)
print(f"nested greedy blend over {len(members)} members   ({time.time() - t:.0f}s)")
for k, w, best_nm in PER_FOLD:
    shown = ", ".join(f"{nm} {wt:.3f}" for nm, wt in sorted(w.items(), key=lambda kv: -kv[1]))
    print(f"  fold {k}: best single elsewhere = {best_nm:<12}  weights: {shown}")
chosen_ever = set().union(*[set(w) for _, w, _ in PER_FOLD])
never = [nm for nm in members if nm not in chosen_ever]
print()
print(f"blend            {B_AUC:.6f}")
print(f"best single      {S_AUC:.6f}")
print(f"blend gain       {B_AUC - S_AUC:+.6f}  = "
      f"{(B_AUC - S_AUC) / SE_REP:.3f} replication SE")
print(f"members never chosen on ANY fold: {len(never)} of {len(members)}  {never}")

In [ ]:
t = time.time()
print(f"{'member':<14}{'solo AUC':>11}{'marginal':>12}{'in rep SE':>12}")
print("-" * 49)
LIVE_MARGINAL = {}
for nm in members:
    reduced, _, _ = nested_blend([m for m in members if m != nm])
    LIVE_MARGINAL[nm] = max(B_AUC - reduced, 0.0)
    print(f"{nm:<14}{roc_auc_score(y_r, POOL[nm]):>11.6f}"
          f"{LIVE_MARGINAL[nm]:>12.6f}{LIVE_MARGINAL[nm] / SE_REP:>12.3f}")
print("-" * 49)
n_zero_live = sum(1 for v in LIVE_MARGINAL.values() if v == 0.0)
n_small_live = sum(1 for v in LIVE_MARGINAL.values() if v < 0.05 * SE_REP)
print(f"members whose removal changes the blend by exactly nothing: "
      f"{n_zero_live} of {len(members)}")
print(f"members worth less than 0.05 replication SE (invisible either way): "
      f"{n_small_live} of {len(members)}   ({time.time() - t:.0f}s)")
print()
print(f"FULL RUN,   nine members: {n_zero} of {len(MEMBERS)} worth exactly zero, "
      f"blend gain {in_se(BLEND_MEAN_DELTA):.3f} SE")
print(f"THIS KERNEL, six members: {n_zero_live} of {len(members)} worth exactly zero, "
      f"blend gain {(B_AUC - S_AUC) / SE_REP:.3f} replication SE")

Six members is not nine and a subsample is not the full frame, so this is a check on the
mechanism rather than a reproduction of the table.

**It does not reproduce the headline, and I want to be straight about why.** The live blend
gains much more than the full-run blend does, and fewer of its members price at exactly zero.
Two reasons, and neither is flattering to the live version. Half the live pool is three
near-copies of each other, the ladder arms, rather than nine independently designed families,
so the greedy search has an easier job finding complementary weight. And the live best single
is deliberately undertrained, 200,000 rows and a fixed tree count, which leaves room above it
that does not exist at full scale. A blend looks best when the thing it is beating is weak.

What the live cell does reproduce is the part of the mechanism that matters. The greedy search
puts nearly all its weight on a handful of members; the weakest family is never selected on
any fold and prices at exactly zero; and the count of members worth less than one twentieth of
a standard error is printed rather than asserted. Read the two bottom lines as the honest side
by side. The nine-member table is the one with 668,665 rows behind it.

---

## Part 5: The punchline, in one screen

Everything above collapses into four numbers.

In [ ]:
items = [
    ("one encoding change (rung 0 -> rung 1)",     RUNG[1] - RUNG[0]),
    ("all nine rungs after it (best - worst)",     SPAN_POST),
    ("nine model families blended (over best)",    BLEND_MEAN_DELTA),
    ("the whole ladder (rung 0 -> best rung)",     max(a for _, _, a, _ in LADDER) - RUNG[0]),
]
print(f"Against the reference public standard error of {SE:.6f}:")
print()
print(f"{'':<44}{'gain':>12}{'in SE':>9}{'visible on a board?':>22}")
print("-" * 87)
for label, d in items:
    vis = "yes, barely" if in_se(d) >= 1 else ("marginal" if in_se(d) >= 0.5 else "no")
    print(f"{label:<44}{d:>+12.6f}{in_se(d):>9.3f}{vis:>22}")
print("-" * 87)
share = (RUNG[1] - RUNG[0]) / (max(a for _, _, a, _ in LADDER) - RUNG[0])
print(f"  the first rung is {share:.1%} of the entire ladder")
print(f"  features at the best rung: {BEST_Z[1]}      features at the widest rung: "
      f"{WIDEST[1]}")
print(f"  rung 6 and rung 8 share a schedule and differ only in features:")
print(f"    z6  {LADDER[6][1]} columns  {RUNG[6]:.6f}")
print(f"    z8  {LADDER[8][1]} columns  {RUNG[8]:.6f}   "
      f"{RUNG[8] - RUNG[6]:+.6f} = {in_se(RUNG[8] - RUNG[6]):+.3f} SE for "
      f"{LADDER[8][1] - LADDER[6][1]} more columns")

One change was worth 1.06 standard errors. Everything after it, nine rungs and as many as
220 extra columns, was worth 0.16. Nine model families blended were worth 0.10. And the rung
carrying the most features is the worst of the nine.

1.06 SE is itself only just readable. One standard error is roughly the scale at which a
single 57,000-row public split stops being able to tell you much of anything. So the honest
summary of ten rungs of feature engineering on this board is: one change I can barely see,
and nine I cannot see at all.

If you are stuck at 0.9457 and deciding what to add next, the measured answer is that the
category of thing you are about to add has already been tried nine times and has a median
payoff of about a sixth of a standard error.

---

## Part 6: Why the ceiling is where it is

Everything above is a description. This part is the explanation, and it is the part I would
most like someone to attack.

**The claim.** This is a synthetic frame. A generator was fitted to a small real dataset and
sampled to produce 955,236 rows over thirteen columns. The amount of signal in those thirteen
columns is fixed by the generator. One good encoding extracts nearly all of it; everything
after that is re-reading signal that has already been read. The plateau is a fact about the
**data**, not about my models, which is why adding a tenth model family cannot break it.

That is a strong claim, so here are four measurements from `train.csv` and `test.csv` that
could have refuted it.

### Evidence 1: the shape of the frame, and whether anything can be memorised

In [ ]:
print(f"{'column':<32}{'dtype':>10}{'distinct':>10}{'share of rows':>16}")
print("-" * 68)
card = {}
for c in FEATS:
    nu = int(train[c].nunique(dropna=True))
    card[c] = nu
    print(f"{c:<32}{str(train[c].dtype):>10}{nu:>10}{nu / N_TRAIN:>16.6f}")
print("-" * 68)
NARROW_C = [c for c in FEATS if card[c] <= 50]
WIDE_C = [c for c in FEATS if card[c] > 50]
print(f"  {len(NARROW_C)} of {len(FEATS)} columns have 50 or fewer distinct values")
print(f"  the wide ones are {WIDE_C}")
grid = 1
for c in NARROW_C:
    grid *= card[c]
print(f"  the narrow columns alone span {grid:,} distinct cells")

t = time.time()
dup_train = int(train[FEATS].duplicated().sum())
both = pd.concat([train[FEATS], test[FEATS]], ignore_index=True)
dup_both = int(both.duplicated().sum())
n_union = len(both) - dup_both
shared = (N_TRAIN - dup_train) + (len(test) - int(test[FEATS].duplicated().sum())) - n_union
del both; gc.collect()
print()
print(f"exact feature-row duplicates within train        {dup_train:>10,} of {N_TRAIN:,}")
print(f"exact feature-row duplicates across train+test   {dup_both:>10,} of "
      f"{N_TRAIN + len(test):,}")
print(f"distinct feature patterns appearing in BOTH      {shared:>10,}")
print(f"   ({time.time() - t:.0f}s)")

Two facts that pull in opposite directions, and the tension between them is the whole story.

Eleven of the thirteen columns are narrow: between two and forty-five distinct values each.
Most of this frame is a handful of small categorical and small-integer alphabets. Only income
and commute distance are wide, and those two are what make every row different from every
other row.

And there are **no exact duplicate rows at all**, not inside train, not across train and
test, not one.

So there is nothing to memorise and no lookup table to steal. A model here has no option but
to learn the generator's conditional law, which is exactly the situation in which every
sufficiently flexible model family converges on the same answer. That is already a partial
explanation for Part 2.

### Evidence 2: is test drawn from the same generator as train?

In [ ]:
t = time.time()
adv = pd.concat([train[FEATS], test[FEATS]], ignore_index=True)
for c in CATEG:
    adv[c] = adv[c].astype("category")
is_test = np.r_[np.zeros(N_TRAIN, np.int8), np.ones(len(test), np.int8)]
adv_oof = np.zeros(len(adv))
for tr_i, va_i in StratifiedKFold(3, shuffle=True,
                                 random_state=CFG["SEED"]).split(adv, is_test):
    m = lgb.LGBMClassifier(random_state=CFG["SEED"], n_jobs=-1, objective="binary",
                           n_estimators=300, learning_rate=0.05, num_leaves=63,
                           min_child_samples=40, verbose=-1)
    m.fit(adv.iloc[tr_i], is_test[tr_i])
    adv_oof[va_i] = m.predict_proba(adv.iloc[va_i])[:, 1]
    del m; gc.collect()
ADV_AUC = roc_auc_score(is_test, adv_oof)
del adv, adv_oof; gc.collect()
print(f"adversarial validation, train vs test, 3-fold: AUC {ADV_AUC:.6f}  "
      f"({time.time() - t:.0f}s)")
print(f"  0.500 means the two frames are indistinguishable; 1.000 means trivially separable")
print(f"  measured distance from indistinguishable: {abs(ADV_AUC - 0.5):.6f}")

If train and test came from different passes of the generator, or from different
parameterisations, a classifier would find the difference and this number would rise above
0.5. What it actually printed is above.

This matters for the saturation argument in a specific way: a distribution shift is one of
the few things that would let a *more flexible* model pay off, because a model that adapts to
the test distribution would beat one that does not. Without a shift, that avenue is closed
too.

### Evidence 3: how much of the ceiling does pure counting reach?

In [ ]:
def binned_keys(df, nbins):
    """Each column reduced to an integer key. Wide numerics get quantile bins
    whose edges come from train; everything else keeps its own values."""
    out = {}
    for c in FEATS:
        s = df[c]
        if c in WIDE_C:
            edges = pd.qcut(train[c], nbins, duplicates="drop", retbins=True)[1]
            out[c] = np.digitize(s.to_numpy(float), edges[1:-1])
        elif c in CATEG:
            out[c] = pd.Categorical(s.astype(str), categories=VOCAB[c]).codes
        else:
            out[c] = s.to_numpy()
    return pd.DataFrame(out, index=df.index)

t = time.time()
KEYS = binned_keys(train, CFG["LOOKUP_BINS"])
SMOOTH = CFG["LOOKUP_SMOOTH"]

def logit(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return np.log(p / (1 - p))

per_col, additive = {}, np.zeros(N_TRAIN)
for c in FEATS:
    col = np.zeros(N_TRAIN)
    for k in range(CFG["N_SPLITS"]):
        tr_m, va_m = FOLDS != k, FOLDS == k
        prior = float(y[tr_m].mean())
        g = pd.DataFrame({"k": KEYS[c][tr_m], "y": y[tr_m]}).groupby("k")["y"].agg(
            ["sum", "count"])
        sm = (g["sum"] + SMOOTH * prior) / (g["count"] + SMOOTH)
        col[va_m] = pd.Series(KEYS[c][va_m]).map(sm).fillna(prior).to_numpy()
    per_col[c] = roc_auc_score(y, col)
    additive += logit(col)

ADDITIVE_AUC = roc_auc_score(y, additive)
print(f"Cross-fitted smoothed cell means, one column at a time "
      f"(smooth={SMOOTH:g}, {CFG['LOOKUP_BINS']} bins on the wide columns):")
print()
print(f"{'column':<40}{'AUC alone':>12}")
print("-" * 52)
for c, a in sorted(per_col.items(), key=lambda kv: -kv[1]):
    print(f"{c:<40}{a:>12.6f}")
print("-" * 52)
best_rung_label = f"best full-run rung (z{BEST_Z[0]})"
print(f"{'ADDITIVE over all 13 (sum of logits)':<40}{ADDITIVE_AUC:>12.6f}")
print(f"{'plain LightGBM from Part 1':<40}{AUC_PLAIN:>12.6f}")
print(f"{best_rung_label:<40}{BEST_Z[2]:>12.6f}")
beaten = [t for t, w, a in MEMBERS if a < ADDITIVE_AUC]
print()
print(f"  full-run model families this lookup beats outright: {beaten}")
print(f"   ({time.time() - t:.0f}s)")

In [ ]:
# The staircase. Each level is more machinery than the one above it. The right
# column is what that machinery bought, in reference standard errors.
# One ladder, one baseline. The blend is NOT on this ladder: it was fitted over
# the nine library members, so its baseline is member h and not the best rung.
# Putting it in the same column would have printed a negative step and meant
# nothing, which is exactly the mistake this cell exists to avoid.
STAIRS = [
    ("counting: 13 cross-fitted lookup tables, summed",      ADDITIVE_AUC, "live"),
    ("+ a plain LightGBM on the same 13 raw columns",        AUC_PLAIN,    "live"),
    ("+ digits, interactions, frequency twins (154 cols)",   RUNG[0],      "full run"),
    ("+ dual target encoding (292 cols)",                    RUNG[1],      "full run"),
    ("+ nine more rungs, up to 374 cols, best of them",      BEST_Z[2],    "full run"),
]
print(f"{'level':<52}{'AUC':>11}{'step':>11}{'in SE':>8}  source")
print("-" * 96)
prev = None
for label, auc, src in STAIRS:
    step = "" if prev is None else f"{auc - prev:>+11.6f}"
    se_s = "" if prev is None else f"{in_se(auc - prev):>8.2f}"
    print(f"{label:<52}{auc:>11.6f}{step}{se_s}  {src}")
    prev = auc
print("-" * 96)
print(f"{'and, on its own baseline of member h:':<52}{BLEND_AUC:>11.6f}"
      f"{BLEND_MEAN_DELTA:>+11.6f}{in_se(BLEND_MEAN_DELTA):>8.2f}  full run")
print()
print(f"  counting to the best rung:               "
      f"{BEST_Z[2] - ADDITIVE_AUC:+.6f} = {in_se(BEST_Z[2] - ADDITIVE_AUC):.2f} SE")
print(f"  the last rung of the ladder:             "
      f"{BEST_Z[2] - RUNG[1]:+.6f} = {in_se(BEST_Z[2] - RUNG[1]):.2f} SE")
print(f"  blending nine families on top of h:      "
      f"{BLEND_MEAN_DELTA:+.6f} = {in_se(BLEND_MEAN_DELTA):.2f} SE")
print()
print("  Every step is a fraction of the step above it, and the last two are")
print("  invisible on a leaderboard. That is what a ceiling looks like from")
print("  underneath.")

This is the measurement I would keep if I could keep only one.

There is no model in the additive number. It is thirteen smoothed lookup tables, cross-fitted
across the same folds, summed on the log-odds scale. No trees, no interactions between
columns, no tuning, nothing that could be called learning beyond counting. And it lands close
enough to the weaker members of my nine-family library to be embarrassing.

The staircase is the honest version of the saturation claim, and it is more interesting than
"everything is noise". It is not that nothing helps. It is that **each level of machinery
buys less than the one before it, by a wide margin, and the last two buy nothing you could
see on a leaderboard.** The gap from counting to the best rung is several standard errors and
is entirely real. The gap from one good encoding to everything after it is not.

So the payoff curve is steeply concave, and by the time you arrive at this board most of it
has already been claimed by whichever public notebook you forked.

### Evidence 4: does more data still help?

In [ ]:
t = time.time()
hold = np.where(FOLDS == 0)[0]
pool = np.where(FOLDS != 0)[0]
rng = np.random.default_rng(CFG["SEED"])
curve = []
for frac in CFG["CURVE_FRACTIONS"]:
    n = int(len(pool) * frac)
    idx = pool if frac >= 1.0 else rng.choice(pool, n, replace=False)
    m = lgb.LGBMClassifier(random_state=CFG["SEED"], n_jobs=-1, **PLAIN_PARAMS)
    m.fit(X_PLAIN.iloc[idx], y[idx])
    a = roc_auc_score(y[hold], m.predict_proba(X_PLAIN.iloc[hold])[:, 1])
    curve.append((frac, n, a))
    del m; gc.collect()

print(f"Plain 13-column LightGBM, fixed held-out fold 0 ({len(hold):,} rows), "
      f"growing training set   ({time.time() - t:.0f}s)")
print()
print(f"{'share of pool':>14}{'rows':>10}{'held-out AUC':>15}{'gain on doubling':>19}"
      f"{'in SE':>8}")
print("-" * 68)
prev = None
for frac, n, a in curve:
    step = "" if prev is None else f"{a - prev[2]:>+19.6f}"
    se_s = "" if prev is None else f"{in_se(a - prev[2]):>8.2f}"
    print(f"{frac:>13.0%}{n:>10,}{a:>15.6f}{step}{se_s}")
    prev = (frac, n, a)
print("-" * 68)
STEPS = [curve[i][2] - curve[i - 1][2] for i in range(1, len(curve))]
RATIOS = [STEPS[i + 1] / STEPS[i] for i in range(len(STEPS) - 1)]
DECAY = float(np.mean(RATIOS))
# Sum of the remaining geometric series, if the decay carries on at DECAY.
REMAINING = STEPS[-1] * DECAY / (1 - DECAY) if 0 < DECAY < 1 else float("nan")
print(f"  first doubling bought {STEPS[0]:+.6f} = {in_se(STEPS[0]):.2f} SE")
print(f"  last  doubling bought {STEPS[-1]:+.6f} = {in_se(STEPS[-1]):.2f} SE")
print(f"  each doubling buys about {DECAY:.2f} of what the previous one bought "
      f"(ratios {', '.join(f'{r:.2f}' for r in RATIOS)})")
print(f"  extrapolating that decay over ALL remaining doublings: "
      f"{REMAINING:+.6f} = {in_se(REMAINING):.2f} SE")
print()
print("  So the curve has not gone flat. It is still climbing, and the largest")
print("  lever left on this board is more rows, which the competition will not")
print("  give you. Every lever you CAN pull is smaller than that.")

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot([n for _, n, _ in curve], [a for _, _, a in curve], "-o",
        color="#2f4858", lw=1.7, ms=6)
ax.axhline(curve[-1][2], color="#c1442e", ls="--", lw=1.0,
           label=f"all {curve[-1][1]:,} rows: {curve[-1][2]:.6f}")
ax.axhspan(curve[-1][2] - SE, curve[-1][2] + SE, color="#f0d0c8", alpha=0.5,
           label=f"+/- 1 public SE ({SE:.6f})")
ax.set_xscale("log")
ax.set_xlabel("training rows (log scale)")
ax.set_ylabel("AUC on the held-out fold")
ax.set_title("Does more data still help? Plain 13-column LightGBM, fixed held-out fold",
             fontsize=11)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

This is the measurement that came closest to refuting the whole argument, and it half
succeeded, so here is what it actually says.

The curve is **not flat**. Each doubling still buys something, and the final doubling, from a
quarter of a million rows to half a million, is worth a real fraction of a standard error. What
has happened instead is that the steps are decaying geometrically, each one worth roughly
three-fifths of the one before it. Extrapolating that decay over every remaining doubling
puts the total headroom from unlimited extra rows at about one standard error.

Read that honestly and it is the most useful number in Part 6. **The biggest lever left on
this board is more training data, and the competition will not give you any.** A lever worth
about one standard error that you cannot pull sits above everything in Parts 2 and 3 that you
can: nine more rungs of feature engineering at 0.16 SE, nine model families blended at 0.10.

Two caveats, both pointing the same way. This curve is the plain thirteen-column model, whose
own ceiling is several thousandths below the ladder's, and a model with better features
usually has a flatter curve because its features already encode what more rows would have
taught it. And a geometric extrapolation from four points is an extrapolation.

### What the four measurements add up to

Put them together:

- there is nothing to memorise, so every model has to learn the same conditional law;
- train and test come from the same generator, so adapting to the test distribution is not an
  available edge;
- counting, with no model in it at all, already sits within a fraction of a per cent of the
  ceiling and outscores at least one of my nine model families outright;
- each successive level of machinery buys a small fraction of what the one before it bought,
  with the last two buying nothing a leaderboard could show;
- and the one resource that would still buy something, more rows, is not on sale.

That is the case for the plateau being a property of the generator rather than of my
modelling. It is consistent with all nine families landing inside 0.0064 of each other, with
six of them adding exactly zero, and with the 374-column rung scoring below the 296-column
one.

**Where the argument is weakest.** The learning curve does not rule out sample size outright,
it prices it, and the price it comes back with is not zero. And nothing here rules out the
possibility that some model class I did not try extracts more from the same columns. What I can say is this: the three best of nine different families land within
one standard error of one another, the other six add nothing at all, two hundred and twenty
extra engineered columns move the top by a sixth of a standard error, and a greedy blend over
all nine moves it by a tenth. That is the behaviour of a frame whose signal has been
exhausted, not of one with a trick left in it.

If you have an argument against this, I would genuinely like to read it in the comments. A
counterexample is worth more to me than agreement.

---

## Part 7: Honest limits

Eight, in the order a reviewer would raise them.

**"Exactly zero" is a property of a greedy blender.** The six zeroes are six members the
greedy forward search never selected on any fold, so removing them from the pool changes the
answer by nothing at all, which is where the exactness comes from. A different combiner, a
ridge stack or a logistic meta-learner, would probably hand them small non-zero weights. What
the result establishes is that those six carried nothing the greedy search could use once h, i
and a were present. That is the operational question. It is not a proof that their predictions
are formally redundant, and I would not want the title read that way.

**Parts 2 and 3 are the two tables you cannot check here.** They come from a completed full
run, they are labelled as such, and their recipes are in the cells. Part 4 replicates the
ladder's two ends and the blend's mechanism live at reduced scale, which is the best a
nine-hour kernel can do, but it is not the same as recomputing the nine-member table. Treat
those two tables as the part of this notebook that rests on my word.

**This is cross-validation on one 5-fold split with seed 42.** Every full-run number in Parts
2 and 3 comes from that one partition. A different seed would give different digits.

**CV deltas and leaderboard deltas are different quantities.** I have converted CV gains into
units of a *leaderboard* standard error, which is a translation between two things that are
not the same. It is the right comparison for the question "would I see this on a board", and
it is not a claim that a CV gain of 0.0011 will show up as a public gain of 0.0011.

**A 0.000164 spread could reorder under a different seed.** The ordering of rungs 1 to 9 is
not stable, and I am not defending it. That is the point, not a caveat: if the ordering of
your feature-engineering rungs is seed-dependent, the ordering was never information.

**The full-run ladder inherited early stopping on the validation fold.** Every rung's
out-of-fold number is mildly optimistic in the same direction, which is why the ladder is only
ever read as *differences between rungs*. The Part 4 replication has no early stopping, so it
does not carry that bias.

**These are my nine families, not every possible model.** No transformer-for-tabular, no
AutoGluon, no large neural ensemble, no pretrained tabular model. Six of the nine I did try
contributed zero, which is evidence but not proof that a tenth would too.

**Part 6 is an argument, not a theorem.** Four measurements consistent with saturation are
four measurements consistent with saturation. I cannot compute this generator's Bayes rate and
I have not claimed a number for it.

---

## Part 8: What actually paid

One thing on this board was worth a standard error: **dual target encoding over the numerics,
the categoricals and the quantisation digits, all treated as string categories, fitted inside
the fold with sklearn's `TargetEncoder` at two smoothing levels.** That is rung 1. It took the
frame from 154 columns to 292 and it bought +0.001117, about 1.06 public standard errors.

Everything I did afterwards, the origin-dataset means, the hand flags, the pair encodings,
the slow schedules, eight more model families and a greedy blend over all nine, came to about
a sixth of a standard error between them.

The only thing Part 6 found that is still worth more than that is more training rows, and
there are no more training rows.

So if you are looking at this board deciding what to build next, my measured advice is:

1. get one good encoding of every column, including the digits;
2. spend one afternoon on the training schedule, because slow-and-wide was the only thing
   after rung 1 that paid twice;
3. stop.

The cell in Part 4 is the one to fork. Point `build_arm` at your own feature blocks, keep the
schedule fixed, and read the delta against `SE_REP` rather than against your hopes. If your
block is worth less than a fifth of a standard error, it will not survive contact with a
leaderboard, however good the CV number looks.

And if you find that something here does not reproduce, say so in the comments with the cell
and the number. I would rather correct this notebook than be quietly wrong in it.

**Credit again**, because it matters: the slow-and-wide schedule is najiama's, and the eight
interactions and the frequency encoding under the whole ladder are evgendvorkin's. The
measurements, the folds and the mistakes are mine.

In [ ]:
HEAD = {
    "n_train": N_TRAIN, "n_test": N_TEST,
    "n_public": N_PUBLIC, "n_private": N_PRIVATE,
    "prevalence": round(PREV, 6),
    "plain_oof_auc": round(float(AUC_PLAIN), 6),
    "se_hanley_public": round(hanley_at(auc_full, N_PUBLIC), 6),
    "se_delong_public": round(DELONG_PUB, 6),
    "se_empirical_public": round(SIGMA_PUB, 6),
    "se_reference": SE,
    "first_rung_gain": round(RUNG[1] - RUNG[0], 6),
    "first_rung_in_se": round(in_se(RUNG[1] - RUNG[0]), 3),
    "rungs_1_to_9_span": round(SPAN_POST, 6),
    "rungs_1_to_9_span_in_se": round(in_se(SPAN_POST), 3),
    "blend_gain_in_se": round(in_se(BLEND_MEAN_DELTA), 3),
    "members_worth_zero": n_zero,
    "replication_rows": int(R_ROWS),
    "replication_se": round(SE_REP, 6),
    "replication_R0": round(REP["R0"]["auc"], 6),
    "replication_R1": round(REP["R1"]["auc"], 6),
    "replication_R2": round(REP["R2"]["auc"], 6),
    "live_blend_members": len(members),
    "live_blend_zero_marginal": n_zero_live,
    "live_blend_gain": round(B_AUC - S_AUC, 6),
    "live_blend_gain_in_rep_se": round((B_AUC - S_AUC) / SE_REP, 3),
    "duplicate_rows_train_plus_test": dup_both,
    "adversarial_train_vs_test_auc": round(float(ADV_AUC), 6),
    "additive_lookup_auc": round(float(ADDITIVE_AUC), 6),
    "additive_to_best_rung_in_se": round(in_se(BEST_Z[2] - ADDITIVE_AUC), 2),
    "learning_curve": [[f, n, round(a, 6)] for f, n, a in curve],
    "wall_seconds": round(time.time() - T0, 1),
}
os.makedirs(WORK, exist_ok=True)
np.save(os.path.join(WORK, "folds_seed42.npy"), FOLDS)
np.save(os.path.join(WORK, "oof_plain.npy"), OOF)
with open(os.path.join(WORK, "headline.json"), "w") as fh:
    json.dump(HEAD, fh, indent=1)
for k, v in HEAD.items():
    print(f"  {k:<32} {v}")
print()
print("written: folds_seed42.npy (the shared 5-fold partition, seed 42),")
print("         oof_plain.npy   (the plain 13-column out-of-fold vector),")
print("         headline.json")